### PART 1: SCRAPING GOODREADS (TO ATTAIN ENLIGHTENMENT)

For your homework, you're going to be scraping quotes from famous Sufi poets on Goodreads — how fun!

In [ ]:
# import the libraries you need... 

import csv
import time
import requests
from bs4 import BeautifulSoup

In [ ]:
url = "https://www.goodreads.com/quotes/tag/sufi"

# Your identification
headers = {"user-agent" : "Mozilla/5.0 (Windows NT 6.1) AppleWebKit/537.36(KHTML, like Gecko) Chrome/41.0.2228.0 Safari/537.36;",
"from": "Fariha-Rahman fariha.rahman74@journalism.cuny.edu"}

# Send a request to Wikipedia and pull down the Sufi mystics page
page = requests.get(url, headers=headers, timeout=30)
print("Status code:", page.status_code)
page.raise_for_status()

# Isolate the html from the webpage
page_content = page.content


In [41]:
# Ask BeautifulSoup to parse the html
soup = BeautifulSoup(page_content, "html.parser")

# Find all the quotes in the html and store them in a variable called quotes
quotes = soup.find_all("div", class_="quote")

# Loop through the variable you created and print out the quote text
for quote in quotes:
    quote_box = quote.find("div", class_="quoteText")
    if quote_box:
        quote_text = quote_box.get_text(" ", strip=True).split("―")[0].strip()
        print(quote_text)

“What you seek is seeking you.”
“I know you're tired but come, this is the way.”
“Don't Worry Be Happy”
“Happiness comes from helping others, by being with others, and by sharing, even if it's only a smile.”
“And when your soul, the flame, the spark, meets with the divine fuel that is so pure and so strong, it results in immense enlightenment: the enlightenment of God. Light upon light, Noorun Alaa Noor.”
“Small good decisions will lead you to the glorious path of success.”
“remember that trying is eighty percent of doing something. So try and don't give up, because not giving up is the other twenty percent.”
“It is the message, not the man, which is important to the Sufis.”
“Forget your voice, sing! Forget your feet, dance! Forget your life, live! Forget yourself and be!”
“Love is and should always be unconditional.”
“your life will not end with death. You are Immortal. You were always there and you always will be.”
“Successful people enjoy the journeys they embark on, irrespective of

Let's take it a step further. Your goal is to create a data object that stores the quote and the quote author. But you might have noticed that you have a problem: if you grab all the text from each quote box, you get a big string containing both the quote AND the author. You can isolate the author's name by using soup.find(...) for the class associated with the author name. But how to isolate the quote? 

There is not one solution to this. But a way to go about it is to use string parsing. 

I will give you two functions that you can use to achieve this, but you can use others if you look them up and find that they work. To get rid of the newlines, you can use .replace("\n","") and to divide the block up text by a certain delimiter (hint hint), you can use 
.split(...)

Look up these two functions and how they work in Python if you're unfamiliar with them.
 


In [42]:
# Loop through the variable you created again and, for each block of text, do the following: 
# 1. Remove the newlines
# 2. Divide the string by the delimiter and store the result in a temporary variable. Now you have a list of strings — basically the quote has been batched up. 
# 3. Create the variables quote_text and author_text using the temporary variable you just made. 
# 4. Save the data in a dictionary and add that dictionary to the variable sufi_quotes
# note: you might want to use the string parsing function .replace() again to clean up the author name, since some have commas and other extraneous information after them...

sufi_quotes = []
for quote in quotes:
    quote_box = quote.find("div", class_="quoteText")
    author_box = quote.find("span", class_="authorOrTitle")
    if quote_box and author_box:
        # Remove newlines and split the quote from the author area
        quote_block = quote_box.get_text("|", strip=True)
        quote_parts = quote_block.replace("\n", "").split("|")
        # The first part is the quote text
        quote_text = quote_parts[0].strip().strip("“”")
        # Extract and clean the author name
        author_text = author_box.get_text(" ", strip=True)
        author_text = author_text.replace(",", "").strip()
        # Store both values in a dictionary
        sufi_quote = {
            "quote": quote_text,
            "author": author_text
        }
        sufi_quotes.append(sufi_quote)
# Print the dictionaries to check the results
for item in sufi_quotes:
    print(item)


{'quote': 'What you seek is seeking you.', 'author': 'Mawlana Jalal-al-Din Rumi'}
{'quote': "I know you're tired but come, this is the way.", 'author': "Jalalu'l-din Rumi"}
{'quote': "Don't Worry Be Happy", 'author': 'Meher Baba'}
{'quote': "Happiness comes from helping others, by being with others, and by sharing, even if it's only a smile.", 'author': 'Zain Hashmi'}
{'quote': 'And when your soul, the flame, the spark, meets with the divine fuel that is so pure and so strong, it results in immense enlightenment: the enlightenment of God. Light upon light, Noorun Alaa Noor.', 'author': 'Zain Hashmi'}
{'quote': 'Small good decisions will lead you to the glorious path of success.', 'author': 'Zain Hashmi'}
{'quote': "remember that trying is eighty percent of doing something. So try and don't give up, because not giving up is the other twenty percent.", 'author': 'Zain Hashmi'}
{'quote': 'It is the message, not the man, which is important to the Sufis.', 'author': 'Idries Shah'}
{'quote':

 $\alpha^2$

In [44]:
# Time to save the info to a csv file. Use DictWriter to do this. 
# You can check out the Python DictWriter documentation here: https://docs.python.org/3/library/csv.html#csv.DictWriter
# (hint: you will loop through sufi_quotes to do this!)
filename = "sufi_quotes.csv"
with open(filename, "w", newline="", encoding="utf-8") as csvfile:
    fieldnames = ["quote", "author"]
    writer = csv.DictWriter(csvfile, fieldnames=fieldnames)
    writer.writeheader()
    for item in sufi_quotes:
        writer.writerow(item)
print(f"Saved {len(sufi_quotes)} quotes to {filename}")

Saved 30 quotes to sufi_quotes.csv


Wow, looks like there's 20 pages of Sufi quotes on Goodreads?! How fortunate are we? Let's take it a step further because we're so enchanted by the divine teachings of the mystics...

Click between the different pages and see how the url changes. There's a pattern there.

In [45]:
# Create a variable called sufi_quote_urls 
# Using a loop, populate this variable with every Goodreads url containing Sufi mystic quotes
sufi_quote_urls = []
for page_num in range(1, 21):
    page_url = f"https://www.goodreads.com/quotes/tag/sufi?page={page_num}"
    sufi_quote_urls.append(page_url)
for page_url in sufi_quote_urls:
    print(page_url)

https://www.goodreads.com/quotes/tag/sufi?page=1
https://www.goodreads.com/quotes/tag/sufi?page=2
https://www.goodreads.com/quotes/tag/sufi?page=3
https://www.goodreads.com/quotes/tag/sufi?page=4
https://www.goodreads.com/quotes/tag/sufi?page=5
https://www.goodreads.com/quotes/tag/sufi?page=6
https://www.goodreads.com/quotes/tag/sufi?page=7
https://www.goodreads.com/quotes/tag/sufi?page=8
https://www.goodreads.com/quotes/tag/sufi?page=9
https://www.goodreads.com/quotes/tag/sufi?page=10
https://www.goodreads.com/quotes/tag/sufi?page=11
https://www.goodreads.com/quotes/tag/sufi?page=12
https://www.goodreads.com/quotes/tag/sufi?page=13
https://www.goodreads.com/quotes/tag/sufi?page=14
https://www.goodreads.com/quotes/tag/sufi?page=15
https://www.goodreads.com/quotes/tag/sufi?page=16
https://www.goodreads.com/quotes/tag/sufi?page=17
https://www.goodreads.com/quotes/tag/sufi?page=18
https://www.goodreads.com/quotes/tag/sufi?page=19
https://www.goodreads.com/quotes/tag/sufi?page=20


In [ ]:
all_sufis = []
for page_url in sufi_quote_urls:
    page = requests.get(page_url, headers=headers, timeout=30)
    time.sleep(2)
    

# Loop through your sufi_quote_urls variable. On each iteration of the loop:
# 1. Send a request to the server using requests
# 2. Include the line of code time.sleep(2) after you make a server request. We'll go over what this does next class.
# 3. Parse the page using BeautifulSoup
# 4. Loop through the quotes again, this time just extracting the author name (easier)
# 5. Store the author name the list all_sufis

In [73]:
if page.status_code != 200:
        print("Could not access:", page_url, page.status_code) 
soup = BeautifulSoup(page.content, "html.parser")
quotes = soup.find_all("div", class_="quote")
for quote in quotes:
        author_box = quote.find("span", class_="authorOrTitle")
        if author_box:
            author_text = author_box.get_text(" ", strip=True)
            author_text = author_text.replace(",", "").strip()
            all_sufis.append(author_text)

In [74]:
# Loop through all_sufis and print out the sufi's name
for sufi in all_sufis:
    print(sufi)
print("Total author entries:", len(all_sufis))

Mawlana Jalal-al-Din Rumi
Jalalu'l-din Rumi
Meher Baba
Zain Hashmi
Idries Shah
Kamand Kojouri
Amir Khusrau
Khwaja Shamsuddin Mohammad
Bulleh Shah
Shams-i Tabrizi
Riaz Ahmed Gohar Shahi
ʿAbd Al-Qadir al-Jilani
Qian Tao
andra dobing
Frederic Manning
Taufiq el-Hakim
Llewellyn Vaughan-Lee
Donna Goddard
Laurence Galian
Abhijit Naskar
Attar of Nishapur
Richard Burton
Fakeer Ishavardas
Sir Richard F. Burton
Dinakar  Phillip
Soroosh Shahrivar
A. Helwa
Peter Lamborn Wilson
Elif Shafak
Baraka Blue
Shems Friedlander
Hasnain Waris
Aiyaz Uddin
Shaikh Abu-Saeed Abil-Kheir
Idries Shah Reflections
Fakhruddin Iraqi
Rumi
Ibn al-Husayn al-Sulami
James S. Cutsinger
Hazrat Inayat Khan
Syed Muhammad Naquib al-Attas
Raden M Kamil
M.R. Bawa Muhaiyaddeen
Attar translated by Sholeh Wolpe
William C. Chittick
Kabir Helminski
Barbara Brown Taylor
Kabir Edmund Helminski
Osho
Master Nursi
Ashfaq Ahmed
osho
Farid ud-Din Attar
Neelam Saxena Chandra
Suzy Kassem
Darma Mohammad
Mevlana Rumi
Jean de Cabalis
Shah Asad Rizv

In [70]:
## I want to remove the duplicate names from the list
all_sufis = list(dict.fromkeys(all_sufis))
# Print each unique author name
for sufi in all_sufis:
    print(sufi)
print("Total unique authors:", len(all_sufis))

Mawlana Jalal-al-Din Rumi
Jalalu'l-din Rumi
Meher Baba
Zain Hashmi
Idries Shah
Kamand Kojouri
Amir Khusrau
Khwaja Shamsuddin Mohammad
Bulleh Shah
Shams-i Tabrizi
Riaz Ahmed Gohar Shahi
ʿAbd Al-Qadir al-Jilani
Qian Tao
andra dobing
Frederic Manning
Taufiq el-Hakim
Llewellyn Vaughan-Lee
Donna Goddard
Laurence Galian
Abhijit Naskar
Attar of Nishapur
Richard Burton
Fakeer Ishavardas
Sir Richard F. Burton
Dinakar  Phillip
Soroosh Shahrivar
A. Helwa
Peter Lamborn Wilson
Elif Shafak
Baraka Blue
Shems Friedlander
Hasnain Waris
Aiyaz Uddin
Shaikh Abu-Saeed Abil-Kheir
Idries Shah Reflections
Fakhruddin Iraqi
Rumi
Ibn al-Husayn al-Sulami
James S. Cutsinger
Hazrat Inayat Khan
Syed Muhammad Naquib al-Attas
Raden M Kamil
M.R. Bawa Muhaiyaddeen
Attar translated by Sholeh Wolpe
William C. Chittick
Kabir Helminski
Barbara Brown Taylor
Kabir Edmund Helminski
Osho
Master Nursi
Ashfaq Ahmed
osho
Farid ud-Din Attar
Neelam Saxena Chandra
Suzy Kassem
Darma Mohammad
Mevlana Rumi
Jean de Cabalis
Shah Asad Rizv

### PART 2: THINKING AHEAD TO ASSIGNMENT 1 

For this assignment you will be doing an initial analysis of a large data set. You will deliver two things:

1. One Google Doc with a head line, a dek, and 2-3 paragraphs that summarize your analysis, your methodology and your findings. You are allowed to present your findings in a chart from DataWrapper, too, and will receive extra credit for it. Note that this part of the assignment should be written for a general audience, be brief and be presented in clear and simple terms

2. A data analysis presented in a Github repository which contains your scrapers (if you made any) along with a Jupyter notebook file containing your analysis with Pandas. 

Do not worry about the details of the Github repository now. We will be learning how to make that — and use Pandas — in future classes. I would just like for you to start thinking about what data you want to analyze for this assignment. 

You will receive extra credit if you scrape your data, but you don't have to. You should be working with a large dataset (hundreds of thousands of rows, ideally) and you should have a research question. Please do some research and paste below 1-3 datasets with research questions for each. 

This won't be your official pitch. This is me basically asking you to start thinking about this assignment. Your next homework will require an official pitch, which I will grade and respond to. The assignment due date has been moved up a week — it will be due November 2. 

Please put the links to your data + your research question(s) here:

1: MTA Ridership — How has NYC public transit usage changed since 2020?
Dataset: MTA Daily Ridership and Traffic Data (2020–present) (https://data.ny.gov/d/sayj-mze2)
Research question: How has ridership on New York City's subways, buses, and commuter rail systems changed since 2020, and which modes of transportation have recovered the most?
Why I'm interested: I want to investigate how New Yorkers' public transportation habits have changed since the pandemic. I could use Pandas to compare daily ridership across different transit systems, examine weekday and weekend patterns, and track changes over time. The dataset includes daily ridership estimates for subways, buses, the Long Island Rail Road, Metro-North, and other MTA services, making it useful for analyzing transportation trends.

2: Citi Bike — How has bike-sharing changed in NYC?
Dataset: Citi Bike System Data (https://citibikenyc.com/system-data)
Research question: How does Citi Bike usage differ between subscribers and casual riders, and how do riding patterns change by season and time of day?
Why I'm interested: I want to explore how New Yorkers use bike-sharing throughout the year and whether casual riders and subscribers have different travel habits. I could analyze trip duration, popular stations, time of day, and seasonal patterns. Citi Bike publishes downloadable trip records containing start and end times, station locations, and rider types.

3: NYC Congestion Pricing — Has congestion pricing reduced traffic in Manhattan?
Dataset: MTA Congestion Relief Zone Vehicle Entries: Beginning 2025 (https://data.ny.gov/d/t6yz-b64h)
Research question: How have vehicle entries into Manhattan's Congestion Relief Zone changed since congestion pricing began in January 2025, and do the changes differ by time of day, entry location, and vehicle type?
Why I'm interested: I want to investigate whether congestion pricing has changed how people drive into Manhattan. I could use Pandas to analyze vehicle entries by hour, compare weekdays with weekends, and identify which types of vehicles have experienced the biggest changes. The MTA dataset includes vehicle entries in 10-minute intervals, broken down by crossing location and vehicle class, making it useful for identifying traffic patterns since the program began. I would also compare the data with pre-2025 traffic figures to put the changes into context.